# Session 11 · Logistic Regression II + Decision Boundaries

**Machine Learning Foundations · Sanketana School of Code**

Last session logistic regression handed us a **probability** and quietly cut it at **0.5**. Today we grab that cut — the **threshold** — and move it ourselves. Sliding it up makes the model *stricter*; sliding it down makes it *lenient*. Predictions **flip** as it moves, and the point is to see why.

By the end of this notebook you will be able to:

- move the **threshold** and predict, in advance, how the decisions change
- count how many predictions **flip** when only the cut moves (the model doesn't)
- read logistic regression's **straight-line** decision boundary and watch it slide
- see logistic regression choose among **three** classes, not just two

## Warm-up · Last session's homework

Your coach will walk through Session 10's logistic-vs-KNN comparison (about 10 minutes). Logistic regression won on accuracy *and* returned a probability per student.

Remember the student sitting near **0.5** — the one the model was honestly unsure about? A tiny nudge of the cut would flip them. Today we do the nudging on purpose.

## Step 1 · Build the probabilities once

We fit the logistic `passed` model a single time and keep its **probabilities**. Everything today reuses these same numbers — we only change where we *cut* them.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

students = pd.read_csv("../../../datasets/anchor/student_habits.csv")
habits = ["study_hours_per_week", "attendance_pct", "sleep_hours_per_night",
          "screen_time_hours_per_day", "practice_sessions_per_week"]
X = students[habits].values
y = students["passed"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
model = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), y_train)

# P(pass) for each held-out student — computed ONCE, reused all session.
p_pass = model.predict_proba(scaler.transform(X_test))[:, 1]
print("test students:", len(y_test), " | real passers:", int(y_test.sum()))
print("a few probabilities:", np.round(np.sort(p_pass)[[0, 50, -1]], 3))

## Step 2 · Slide the cut and count the mistakes

Turn the probabilities into pass/fail at three different thresholds. At each cut, count two kinds of mistake:

- **missed passers** — real passers we predicted would fail
- **false passes** — non-passers we predicted would pass

**Predict first:** as the threshold goes *up*, which mistake goes up and which goes down?

In [ ]:
def counts_at(threshold):
    pred = (p_pass >= threshold).astype(int)
    predicted_pass = int(pred.sum())
    missed_passers = int(np.sum((y_test == 1) & (pred == 0)))   # real pass, said fail
    false_passes   = int(np.sum((y_test == 0) & (pred == 1)))   # real fail, said pass
    accuracy = float((pred == y_test).mean())
    return predicted_pass, missed_passers, false_passes, accuracy

print(f"{"cut":>4} {"pred-pass":>10} {"missed":>8} {"false-pass":>11} {"acc":>7}")
for t in [0.3, 0.5, 0.7]:
    pp, miss, fp, acc = counts_at(t)
    print(f"{t:>4} {pp:>10} {miss:>8} {fp:>11} {acc:>7.3f}")

**Read it.** A **low** cut (0.3) is lenient — it catches nearly every real passer (few *missed*) but waves through many who won't pass (many *false passes*). A **high** cut (0.7) is strict — the opposite. Notice accuracy is highest at **0.5** here, yet we might still move the cut if one mistake mattered more. *That's the whole idea.*

## Step 3 · How many students actually flip?

Same probabilities, different cuts — so some students change their predicted outcome. How many flip between a lenient cut and a strict one?

In [ ]:
low  = (p_pass >= 0.3).astype(int)
high = (p_pass >= 0.7).astype(int)
flipped = int(np.sum(low != high))
print(f"{flipped} of {len(p_pass)} students flip between cut=0.3 and cut=0.7")
print("The model never changed — only the line we drew across its probabilities did.")

## Step 4 · The decision boundary is a straight line — and it slides

With two features we can *see* the boundary. Unlike Session 9's wiggly KNN border, logistic regression's boundary is a **straight line**. Moving the threshold **slides that line** across the plot.

In [ ]:
two = ["study_hours_per_week", "attendance_pct"]
X2 = students[two].values
X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y, test_size=0.25, random_state=42, stratify=y)
sc2 = StandardScaler().fit(X2_tr)
m2 = LogisticRegression(max_iter=1000).fit(sc2.transform(X2_tr), y2_tr)

x0 = np.linspace(X2[:, 0].min() - 1, X2[:, 0].max() + 1, 300)
x1 = np.linspace(X2[:, 1].min() - 2, X2[:, 1].max() + 2, 300)
gx, gy = np.meshgrid(x0, x1)
grid = np.c_[gx.ravel(), gy.ravel()]
prob_grid = m2.predict_proba(sc2.transform(grid))[:, 1].reshape(gx.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, thr in zip(axes, [0.5, 0.7]):
    ax.contourf(gx, gy, (prob_grid >= thr).astype(int),
                alpha=0.25, levels=[-0.5, 0.5, 1.5], cmap="coolwarm")
    ax.contour(gx, gy, prob_grid, levels=[thr], colors="k", linewidths=2)
    ax.scatter(X2[:, 0], X2[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=16)
    ax.set_xlabel("study hours per week"); ax.set_ylabel("attendance %")
    ax.set_title(f"threshold = {thr}  (blue = predict pass)")
plt.tight_layout(); plt.show()
print("Same straight boundary, shifted: the stricter cut (0.7) shrinks the blue 'pass' region.")

## Step 5 · More than two classes

Classification isn't only yes/no. Sort students into **three** grade bands — *fail*, *pass*, *distinction* — and logistic regression gives **one probability per class** (summing to 1); the prediction is the **highest** one.

In [ ]:
bands = pd.cut(students["test_score"], bins=[0, 50, 75, 100],
               labels=["fail", "pass", "distinction"])
print("band counts:", bands.value_counts().to_dict())

yb = bands.values
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
    X, yb, test_size=0.25, random_state=42, stratify=yb)
scb = StandardScaler().fit(Xb_tr)
multi = LogisticRegression(max_iter=1000).fit(scb.transform(Xb_tr), yb_tr)
print("multi-class test accuracy:", round(multi.score(scb.transform(Xb_te), yb_te), 3))

row = multi.predict_proba(scb.transform(Xb_te))[0]
print("classes:       ", list(multi.classes_))
print("one student's P:", np.round(row, 2), " -> sums to", round(float(row.sum()), 2))
print("predicted band:", multi.predict(scb.transform(Xb_te))[0])

### ✏️ Your turn

Suppose the school will privately warn any student the model predicts will **fail**, so they can get help early. Would you set the pass/fail threshold **higher or lower** than 0.5, and which mistake are you choosing to make more of? (Two or three sentences.)

*Your answer here:*

## Wrap-up

- The **threshold** turns a probability into a decision; **0.5 is only the default**.
- Moving it **doesn't retrain the model** — it makes the same model **stricter or more lenient**, trading one mistake for the other.
- Logistic regression's boundary is a **straight line** that **slides** as the cut moves (KNN's wiggled — shape reflects the model).
- Logistic regression handles **more than two classes**: a probability each, pick the highest.

**Next session:** we finally **name** the two mistakes (false positive, false negative), organise them into a **confusion matrix**, and — on imbalanced fraud data — discover that plain **accuracy hides the mistake that matters**. That's where *which mistake is worse* becomes *who pays.*

*Homework: a threshold table and a multi-class model, in `homework.ipynb`.*